# Component-Level Trimmed-Away Excess Contributions

This notebook decomposes the excess of official headline PCE inflation over trimmed mean PCE inflation into component-level trimmed-away contributions.

For each of the 178 PCE components, the monthly contribution is:

`trimmed_weight * (price_change - trimmed_monthly_change)`

Components that are not trimmed away in a month remain in the panel with a zero monthly contribution. The official headline reconciliation wedge is kept as an explicit `COVERAGE_RESIDUAL` pseudo-component rather than being allocated across the 178 real components.

The main 12-month measure uses a deterministic Aumann-Shapley allocation over 179 paths: 178 real components plus the coverage residual. Decimal-rate columns are the source of truth; percentage-point helper columns are included for reporting.


In [1]:
from pathlib import Path
import json
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

COMPONENT_AUDIT_PATH = Path('outputs/component_monthly_audit.csv')
TRIMMED_RATES_PATH = Path('outputs/trimmed_mean_rates.csv')

OUTPUT_LONG = Path('outputs/trimmed_away_component_excess_contributions_long.csv')
OUTPUT_TOTALS = Path('outputs/trimmed_away_component_excess_contribution_totals.csv')

FRED_API_KEY = '51f3ac7bc8b65cb6bb2589fc570292be'
FRED_SERIES_ID = 'PCEPI'
COVERAGE_COMPONENT_ID = 'COVERAGE_RESIDUAL'
REAL_COMPONENT_COUNT = 178

for path in [COMPONENT_AUDIT_PATH, TRIMMED_RATES_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f'Component audit input: {COMPONENT_AUDIT_PATH}')
print(f'Trimmed rates input:   {TRIMMED_RATES_PATH}')
print(f'Official headline:     FRED {FRED_SERIES_ID}')


Component audit input: outputs\component_monthly_audit.csv
Trimmed rates input:   outputs\trimmed_mean_rates.csv
Official headline:     FRED PCEPI


In [2]:
component_audit = pd.read_csv(COMPONENT_AUDIT_PATH, parse_dates=['date'])
trimmed_rates = pd.read_csv(TRIMMED_RATES_PATH, parse_dates=['date'])

required_audit_columns = {
    'date', 'period', 'line_item', 'code', 'description', 'price_change',
    'raw_weight', 'retained_weight', 'trimmed_weight', 'lower_trimmed_weight',
    'upper_trimmed_weight', 'trim_side', 'included', 'trimmed_away'
}
required_rate_columns = {'date', 'period', 'trimmed_monthly_change'}
missing_audit_columns = required_audit_columns.difference(component_audit.columns)
missing_rate_columns = required_rate_columns.difference(trimmed_rates.columns)
if missing_audit_columns:
    raise ValueError(f'component_monthly_audit.csv is missing columns: {sorted(missing_audit_columns)}')
if missing_rate_columns:
    raise ValueError(f'trimmed_mean_rates.csv is missing columns: {sorted(missing_rate_columns)}')

component_audit = component_audit.sort_values(['date', 'line_item']).reset_index(drop=True)
trimmed_rates = trimmed_rates.sort_values('date').reset_index(drop=True)

row_counts = component_audit.groupby('date').size()
component_counts = component_audit.groupby('date')['line_item'].nunique()
if not (row_counts.eq(REAL_COMPONENT_COUNT).all() and component_counts.eq(REAL_COMPONENT_COUNT).all()):
    bad_dates = pd.DataFrame({'rows': row_counts, 'components': component_counts})
    bad_dates = bad_dates[(bad_dates['rows'] != REAL_COMPONENT_COUNT) | (bad_dates['components'] != REAL_COMPONENT_COUNT)]
    raise ValueError(f'Expected exactly {REAL_COMPONENT_COUNT} component rows per date; examples:\n{bad_dates.head()}')

print(f'component_audit rows: {len(component_audit):,}')
print(f'trimmed_rates rows:   {len(trimmed_rates):,}')
print(f'date range:           {component_audit["date"].min().date()} to {component_audit["date"].max().date()}')
print(f'real components/date: {REAL_COMPONENT_COUNT}')


component_audit rows: 101,104
trimmed_rates rows:   568
date range:           1979-01-01 to 2026-04-01
real components/date: 178


In [3]:
def fetch_fred_series(series_id, api_key, observation_start):
    params = urllib.parse.urlencode({
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'observation_start': observation_start,
    })
    url = f'https://api.stlouisfed.org/fred/series/observations?{params}'
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = json.load(response)

    if 'error_code' in payload:
        raise RuntimeError(f"FRED error {payload.get('error_code')}: {payload.get('error_message')}")
    if 'observations' not in payload:
        raise RuntimeError(f'Unexpected FRED response keys: {sorted(payload)}')

    fred = pd.DataFrame(payload['observations'])
    fred['date'] = pd.to_datetime(fred['date'])
    fred['value'] = pd.to_numeric(fred['value'].replace('.', np.nan), errors='coerce')
    fred = fred.dropna(subset=['value']).sort_values('date').reset_index(drop=True)
    return fred[['date', 'value']].rename(columns={'value': 'official_pcepi_index'})

local_start = trimmed_rates['date'].min()
fred_observation_start = (local_start - pd.DateOffset(months=1)).strftime('%Y-%m-%d')
official_pcepi = fetch_fred_series(FRED_SERIES_ID, FRED_API_KEY, fred_observation_start)
official_pcepi['official_headline_monthly_change'] = official_pcepi['official_pcepi_index'].pct_change()

official_headline = official_pcepi[official_pcepi['date'].isin(trimmed_rates['date'])].copy()
missing_official_dates = sorted(set(trimmed_rates['date']) - set(official_headline['date']))
if missing_official_dates:
    raise ValueError(f'Missing FRED {FRED_SERIES_ID} observations for local dates: {missing_official_dates[:5]}')
if official_headline['official_headline_monthly_change'].isna().any():
    bad_dates = official_headline.loc[official_headline['official_headline_monthly_change'].isna(), 'date'].dt.strftime('%Y-%m-%d').tolist()
    raise ValueError(f'Missing official monthly changes for local dates: {bad_dates[:5]}')

print(f'FRED observations fetched: {len(official_pcepi):,}')
print(f'FRED date range:           {official_pcepi["date"].min().date()} to {official_pcepi["date"].max().date()}')
print(official_headline.tail().to_string(index=False))


FRED observations fetched: 569
FRED date range:           1978-12-01 to 2026-04-01
      date  official_pcepi_index  official_headline_monthly_change
2025-12-01               128.576                          0.003309
2026-01-01               129.002                          0.003313
2026-02-01               129.520                          0.004015
2026-03-01               130.381                          0.006648
2026-04-01               130.902                          0.003996


In [4]:
component_panel = component_audit.merge(
    trimmed_rates[['date', 'trimmed_monthly_change']],
    how='left',
    on='date',
    validate='many_to_one',
)
if component_panel['trimmed_monthly_change'].isna().any():
    raise ValueError('Missing trimmed monthly change after merging component audit to trimmed rates')

component_panel['component_id'] = component_panel['line_item'].astype('int64').astype(str)
component_panel['is_coverage_residual'] = False
component_panel['monthly_excess_contribution'] = (
    component_panel['trimmed_weight']
    * (component_panel['price_change'] - component_panel['trimmed_monthly_change'])
)

component_headline = (
    component_audit
    .assign(weighted_price_change=lambda df: df['raw_weight'] * df['price_change'])
    .groupby('date', as_index=False)['weighted_price_change']
    .sum()
    .rename(columns={'weighted_price_change': 'component_headline_monthly_change'})
)

monthly_base = (
    trimmed_rates[['date', 'period', 'trimmed_monthly_change']]
    .merge(component_headline, how='left', on='date', validate='one_to_one')
    .merge(official_headline[['date', 'official_pcepi_index', 'official_headline_monthly_change']], how='left', on='date', validate='one_to_one')
    .sort_values('date')
    .reset_index(drop=True)
)
monthly_base['component_total_monthly_excess_contribution'] = (
    component_panel.groupby('date')['monthly_excess_contribution'].sum().reindex(monthly_base['date']).to_numpy()
)
monthly_base['coverage_residual_monthly_excess_contribution'] = (
    monthly_base['official_headline_monthly_change']
    - monthly_base['component_headline_monthly_change']
)
monthly_base['official_total_monthly_excess_contribution'] = (
    monthly_base['component_total_monthly_excess_contribution']
    + monthly_base['coverage_residual_monthly_excess_contribution']
)
monthly_base['component_monthly_identity_residual'] = (
    monthly_base['component_total_monthly_excess_contribution']
    - (monthly_base['component_headline_monthly_change'] - monthly_base['trimmed_monthly_change'])
)
monthly_base['official_monthly_identity_residual'] = (
    monthly_base['official_total_monthly_excess_contribution']
    - (monthly_base['official_headline_monthly_change'] - monthly_base['trimmed_monthly_change'])
)

print(monthly_base.head().to_string(index=False))
print('\nMonthly residuals:')
print(f"component identity max abs: {monthly_base['component_monthly_identity_residual'].abs().max():.3e}")
print(f"official identity max abs:  {monthly_base['official_monthly_identity_residual'].abs().max():.3e}")


      date  period  trimmed_monthly_change  component_headline_monthly_change  official_pcepi_index  official_headline_monthly_change  component_total_monthly_excess_contribution  coverage_residual_monthly_excess_contribution  official_total_monthly_excess_contribution  component_monthly_identity_residual  official_monthly_identity_residual
1979-01-01 1979-01                0.006818                           0.007643                33.597                          0.007557                                     0.000826                                      -0.000086                                    0.000740                         5.724587e-17                        5.724587e-17
1979-02-01 1979-02                0.006177                           0.005387                33.776                          0.005328                                    -0.000790                                      -0.000059                                   -0.000849                         5.898060e-17        

In [5]:
real_output_columns = [
    'date', 'period', 'component_id', 'line_item', 'code', 'description', 'is_coverage_residual',
    'price_change', 'raw_weight', 'retained_weight', 'trimmed_weight', 'lower_trimmed_weight', 'upper_trimmed_weight',
    'trim_side', 'included', 'trimmed_away', 'monthly_excess_contribution'
]
real_long = component_panel[real_output_columns].copy()
real_long['component_sort_key'] = real_long['line_item'].astype(float)

residual_long = monthly_base[['date', 'period', 'coverage_residual_monthly_excess_contribution']].copy()
residual_long = residual_long.rename(columns={'coverage_residual_monthly_excess_contribution': 'monthly_excess_contribution'})
residual_long['component_id'] = COVERAGE_COMPONENT_ID
residual_long['line_item'] = pd.NA
residual_long['code'] = COVERAGE_COMPONENT_ID
residual_long['description'] = 'Official headline coverage residual'
residual_long['is_coverage_residual'] = True
for column in ['price_change', 'raw_weight', 'retained_weight', 'trimmed_weight', 'lower_trimmed_weight', 'upper_trimmed_weight']:
    residual_long[column] = np.nan
residual_long['trim_side'] = 'coverage_residual'
residual_long['included'] = False
residual_long['trimmed_away'] = False
residual_long['component_sort_key'] = REAL_COMPONENT_COUNT + 1
residual_long = residual_long[real_output_columns + ['component_sort_key']]

long = pd.concat([real_long, residual_long], ignore_index=True)
long = long.sort_values(['date', 'component_sort_key']).reset_index(drop=True)
long['monthly_excess_contribution_pp'] = long['monthly_excess_contribution'] * 100.0
long['rolling_12m_excess_contribution'] = (
    long.sort_values(['component_id', 'date'])
    .groupby('component_id', sort=False)['monthly_excess_contribution']
    .transform(lambda series: series.rolling(window=12, min_periods=12).sum())
)
long['rolling_12m_excess_contribution_pp'] = long['rolling_12m_excess_contribution'] * 100.0

print(f'Long rows: {len(long):,}')
print(f'Components including residual: {long["component_id"].nunique():,}')
print(long.head().to_string(index=False))


Long rows: 101,672
Components including residual: 179
      date  period component_id line_item   code        description  is_coverage_residual  price_change  raw_weight  retained_weight  trimmed_weight  lower_trimmed_weight  upper_trimmed_weight trim_side  included  trimmed_away  monthly_excess_contribution  component_sort_key  monthly_excess_contribution_pp  rolling_12m_excess_contribution  rolling_12m_excess_contribution_pp
1979-01-01 1979-01            7         7 DNDCRG New domestic autos                 False      0.004982    0.025674         0.025674        0.000000              0.000000                   0.0      none      True         False                    -0.000000                 7.0                       -0.000000                              NaN                                 NaN
1979-01-01 1979-01            8         8 DNFCRG  New foreign autos                 False      0.004999    0.007733         0.007733        0.000000              0.000000                   0.0

In [6]:
def compound_rate(monthly_path):
    monthly_path = np.asarray(monthly_path, dtype=float)
    return float(np.prod(1.0 + monthly_path) - 1.0)

def aumann_shapley_window(trimmed_path, factor_matrix, nodes, weights):
    total_gap_path = factor_matrix.sum(axis=1)
    allocation = np.zeros(factor_matrix.shape[1], dtype=float)
    for node, weight in zip(nodes, weights):
        state = 1.0 + trimmed_path + node * total_gap_path
        product_all = np.prod(state)
        derivative_by_month = product_all / state
        allocation += weight * (factor_matrix.T @ derivative_by_month)
    return allocation

real_component_ids = (
    component_audit[['line_item']]
    .drop_duplicates()
    .sort_values('line_item')['line_item']
    .astype('int64')
    .astype(str)
    .tolist()
)
component_ids = [*real_component_ids, COVERAGE_COMPONENT_ID]
ordered_dates = monthly_base['date'].tolist()

factor_paths = (
    long.pivot(index='date', columns='component_id', values='monthly_excess_contribution')
    .reindex(index=ordered_dates, columns=component_ids)
)
if factor_paths.isna().any().any():
    raise ValueError('Unexpected missing values in component factor path matrix')

trimmed_path_all = monthly_base['trimmed_monthly_change'].to_numpy(dtype=float)
factor_matrix_all = factor_paths.to_numpy(dtype=float)

nodes_raw, weights_raw = np.polynomial.legendre.leggauss(6)
gauss_nodes = (nodes_raw + 1.0) / 2.0
gauss_weights = weights_raw / 2.0

as_matrix = np.full_like(factor_matrix_all, np.nan, dtype=float)
trimmed_12m = np.full(len(monthly_base), np.nan, dtype=float)
component_headline_12m = np.full(len(monthly_base), np.nan, dtype=float)
official_headline_12m = np.full(len(monthly_base), np.nan, dtype=float)

for end_pos in range(11, len(monthly_base)):
    window_slice = slice(end_pos - 11, end_pos + 1)
    trimmed_window = trimmed_path_all[window_slice]
    factor_window = factor_matrix_all[window_slice, :]
    real_factor_window = factor_window[:, :REAL_COMPONENT_COUNT]

    as_matrix[end_pos, :] = aumann_shapley_window(trimmed_window, factor_window, gauss_nodes, gauss_weights)
    trimmed_12m[end_pos] = compound_rate(trimmed_window)
    component_headline_12m[end_pos] = compound_rate(trimmed_window + real_factor_window.sum(axis=1))
    official_headline_12m[end_pos] = compound_rate(trimmed_window + factor_window.sum(axis=1))

aumann_shapley_long = (
    pd.DataFrame(as_matrix, index=ordered_dates, columns=component_ids)
    .rename_axis('date')
    .reset_index()
    .melt(id_vars='date', var_name='component_id', value_name='aumann_shapley_12m_excess_contribution')
)
long = long.merge(aumann_shapley_long, how='left', on=['date', 'component_id'], validate='one_to_one')
long['aumann_shapley_12m_excess_contribution_pp'] = long['aumann_shapley_12m_excess_contribution'] * 100.0

monthly_base['trimmed_12m_change'] = trimmed_12m
monthly_base['component_headline_12m_change'] = component_headline_12m
monthly_base['official_headline_12m_change'] = official_headline_12m

print(long.loc[long['date'].eq(long['date'].max()), [
    'date', 'component_id', 'description', 'monthly_excess_contribution',
    'rolling_12m_excess_contribution', 'aumann_shapley_12m_excess_contribution'
]].sort_values('aumann_shapley_12m_excess_contribution', key=lambda s: s.abs(), ascending=False).head(10).to_string(index=False))


      date      component_id                                            description  monthly_excess_contribution  rolling_12m_excess_contribution  aumann_shapley_12m_excess_contribution
2026-04-01               115                          Gasoline and other motor fuel                     0.001156                         0.004656                                0.004777
2026-04-01               258 Financial service charges, fees, and commissions (108)                    -0.000479                         0.001859                                0.001912
2026-04-01                50                      Computer software and accessories                     0.000526                         0.001208                                0.001240
2026-04-01               255                                       Commercial banks                     0.000051                         0.001021                                0.001050
2026-04-01               207                                Air transp

In [7]:
def sum_by_date(df, value_column, mask=None):
    source = df if mask is None else df.loc[mask]
    return source.groupby('date')[value_column].sum(min_count=1).reindex(monthly_base['date']).to_numpy()

real_mask = ~long['is_coverage_residual']
residual_mask = long['is_coverage_residual']

monthly_base['component_total_12m_rolling'] = sum_by_date(long, 'rolling_12m_excess_contribution', real_mask)
monthly_base['coverage_residual_12m_rolling'] = sum_by_date(long, 'rolling_12m_excess_contribution', residual_mask)
monthly_base['official_total_12m_rolling'] = sum_by_date(long, 'rolling_12m_excess_contribution')
monthly_base['component_total_12m_aumann_shapley'] = sum_by_date(long, 'aumann_shapley_12m_excess_contribution', real_mask)
monthly_base['coverage_residual_12m_aumann_shapley'] = sum_by_date(long, 'aumann_shapley_12m_excess_contribution', residual_mask)
monthly_base['official_total_12m_aumann_shapley'] = sum_by_date(long, 'aumann_shapley_12m_excess_contribution')

monthly_base['official_12m_gap'] = monthly_base['official_headline_12m_change'] - monthly_base['trimmed_12m_change']
monthly_base['component_monthly_groupby_residual'] = (
    sum_by_date(long, 'monthly_excess_contribution', real_mask)
    - monthly_base['component_total_monthly_excess_contribution']
)
monthly_base['official_monthly_groupby_residual'] = (
    sum_by_date(long, 'monthly_excess_contribution')
    - monthly_base['official_total_monthly_excess_contribution']
)
monthly_base['official_12m_aumann_shapley_identity_residual'] = (
    monthly_base['official_total_12m_aumann_shapley'] - monthly_base['official_12m_gap']
)
monthly_base['official_12m_aumann_shapley_groupby_residual'] = (
    sum_by_date(long, 'aumann_shapley_12m_excess_contribution')
    - monthly_base['official_total_12m_aumann_shapley']
)

pp_columns_long = [
    'monthly_excess_contribution',
    'rolling_12m_excess_contribution',
    'aumann_shapley_12m_excess_contribution',
]
for column in pp_columns_long:
    pp_column = f'{column}_pp'
    if pp_column not in long.columns:
        long[pp_column] = long[column] * 100.0

pp_columns_totals = [
    'trimmed_monthly_change', 'component_headline_monthly_change', 'official_headline_monthly_change',
    'component_total_monthly_excess_contribution', 'coverage_residual_monthly_excess_contribution', 'official_total_monthly_excess_contribution',
    'trimmed_12m_change', 'component_headline_12m_change', 'official_headline_12m_change', 'official_12m_gap',
    'component_total_12m_rolling', 'coverage_residual_12m_rolling', 'official_total_12m_rolling',
    'component_total_12m_aumann_shapley', 'coverage_residual_12m_aumann_shapley', 'official_total_12m_aumann_shapley',
]
for column in pp_columns_totals:
    monthly_base[f'{column}_pp'] = monthly_base[column] * 100.0

first_11_dates = monthly_base['date'].iloc[:11]
remaining_dates = monthly_base['date'].iloc[11:]
first_11_long = long['date'].isin(first_11_dates)
remaining_long = long['date'].isin(remaining_dates)

validation = {
    'real_components_per_date_min': int(component_audit.groupby('date')['line_item'].nunique().min()),
    'real_components_per_date_max': int(component_audit.groupby('date')['line_item'].nunique().max()),
    'missing_official_headline_monthly_changes': int(monthly_base['official_headline_monthly_change'].isna().sum()),
    'component_monthly_identity_max_abs_residual': float(monthly_base['component_monthly_identity_residual'].abs().max()),
    'official_monthly_identity_max_abs_residual': float(monthly_base['official_monthly_identity_residual'].abs().max()),
    'component_monthly_groupby_max_abs_residual': float(monthly_base['component_monthly_groupby_residual'].abs().max()),
    'official_monthly_groupby_max_abs_residual': float(monthly_base['official_monthly_groupby_residual'].abs().max()),
    'official_12m_aumann_shapley_identity_max_abs_residual': float(monthly_base['official_12m_aumann_shapley_identity_residual'].dropna().abs().max()),
    'official_12m_aumann_shapley_groupby_max_abs_residual': float(monthly_base['official_12m_aumann_shapley_groupby_residual'].dropna().abs().max()),
    'first_11_long_12m_values_all_nan': bool(long.loc[first_11_long, ['rolling_12m_excess_contribution', 'aumann_shapley_12m_excess_contribution']].isna().all().all()),
    'remaining_long_12m_values_no_nan': bool(long.loc[remaining_long, ['rolling_12m_excess_contribution', 'aumann_shapley_12m_excess_contribution']].notna().all().all()),
}

TOLERANCE = 1e-10
assert validation['real_components_per_date_min'] == REAL_COMPONENT_COUNT
assert validation['real_components_per_date_max'] == REAL_COMPONENT_COUNT
assert validation['missing_official_headline_monthly_changes'] == 0
assert validation['component_monthly_identity_max_abs_residual'] < TOLERANCE
assert validation['official_monthly_identity_max_abs_residual'] < TOLERANCE
assert validation['component_monthly_groupby_max_abs_residual'] < TOLERANCE
assert validation['official_monthly_groupby_max_abs_residual'] < TOLERANCE
assert validation['official_12m_aumann_shapley_identity_max_abs_residual'] < TOLERANCE
assert validation['official_12m_aumann_shapley_groupby_max_abs_residual'] < TOLERANCE
assert validation['first_11_long_12m_values_all_nan']
assert validation['remaining_long_12m_values_no_nan']

validation_summary = pd.DataFrame({'check': validation.keys(), 'value': validation.values()})
print(validation_summary.to_string(index=False))
print('\nLatest totals:')
print(monthly_base.tail(3).to_string(index=False))


                                                check value
                         real_components_per_date_min   178
                         real_components_per_date_max   178
            missing_official_headline_monthly_changes     0
          component_monthly_identity_max_abs_residual   0.0
           official_monthly_identity_max_abs_residual   0.0
           component_monthly_groupby_max_abs_residual   0.0
            official_monthly_groupby_max_abs_residual   0.0
official_12m_aumann_shapley_identity_max_abs_residual   0.0
 official_12m_aumann_shapley_groupby_max_abs_residual   0.0
                     first_11_long_12m_values_all_nan  True
                     remaining_long_12m_values_no_nan  True

Latest totals:
      date  period  trimmed_monthly_change  component_headline_monthly_change  official_pcepi_index  official_headline_monthly_change  component_total_monthly_excess_contribution  coverage_residual_monthly_excess_contribution  official_total_monthly_excess_contrib

In [8]:
long_output_columns = [
    'date', 'period', 'component_id', 'line_item', 'code', 'description', 'is_coverage_residual',
    'price_change', 'raw_weight', 'retained_weight', 'trimmed_weight', 'lower_trimmed_weight', 'upper_trimmed_weight',
    'trim_side', 'included', 'trimmed_away',
    'monthly_excess_contribution', 'monthly_excess_contribution_pp',
    'rolling_12m_excess_contribution', 'rolling_12m_excess_contribution_pp',
    'aumann_shapley_12m_excess_contribution', 'aumann_shapley_12m_excess_contribution_pp',
]
totals_output_columns = [
    'date', 'period', 'trimmed_monthly_change', 'component_headline_monthly_change', 'official_headline_monthly_change', 'official_pcepi_index',
    'component_total_monthly_excess_contribution', 'coverage_residual_monthly_excess_contribution', 'official_total_monthly_excess_contribution',
    'trimmed_12m_change', 'component_headline_12m_change', 'official_headline_12m_change', 'official_12m_gap',
    'component_total_12m_rolling', 'coverage_residual_12m_rolling', 'official_total_12m_rolling',
    'component_total_12m_aumann_shapley', 'coverage_residual_12m_aumann_shapley', 'official_total_12m_aumann_shapley',
    'component_monthly_identity_residual', 'official_monthly_identity_residual',
    'component_monthly_groupby_residual', 'official_monthly_groupby_residual',
    'official_12m_aumann_shapley_identity_residual', 'official_12m_aumann_shapley_groupby_residual',
    *[f'{column}_pp' for column in pp_columns_totals],
]

long_output = long[long_output_columns].copy()
totals_output = monthly_base[totals_output_columns].copy()

OUTPUT_LONG.parent.mkdir(parents=True, exist_ok=True)
long_output.to_csv(OUTPUT_LONG, index=False)
totals_output.to_csv(OUTPUT_TOTALS, index=False)

print(f'Wrote {OUTPUT_LONG} ({len(long_output):,} rows)')
print(f'Wrote {OUTPUT_TOTALS} ({len(totals_output):,} rows)')


Wrote outputs\trimmed_away_component_excess_contributions_long.csv (101,672 rows)
Wrote outputs\trimmed_away_component_excess_contribution_totals.csv (568 rows)
